# homography: ArUco corners -> top-down warp + 8x8 grid (M1)

The board has four ArUco markers (DICT_4X4_50, ids 0..3 next to a1, h1, h8, a8) printed in
the frame around the playing area. Their positions on the board are known from
`gambit_bringup/config/board.yaml`, so detecting them in the camera image gives four
point correspondences -> a homography from the image to a canonical top-down view
("warp"). In the warp every square is an axis-aligned `square_px` x `square_px` cell, so
the 8x8 grid and the square crops (`capture.ipynb`) are pure arithmetic.

Coordinate frames used here:
* **board** (meters): origin at the outer corner of a1 of the playing area, `u` along the
  files (a -> h), `v` along the ranks (1 -> 8). The playing area is `[0, 8s] x [0, 8s]`.
* **warp** (pixels): the top-down image, `8 * square_px + 2 * margin_px` square. With
  `view: black_bottom` (default, camera on the black side) rank 8 is at the bottom and the
  a-file on the right, like the camera sees it.
* **image** (pixels): the raw (ideally undistorted) camera frame.

Milestone: M1 (calibration + grid overlay). Deliverable: grid overlay on the live image and
the square-center pixel error.

**TODO**
- [ ] fallback when only 3 markers are visible (a hand covers one): affine from 3 points,
      or reuse the last good H (the camera is fixed)
- [ ] use all 16 marker corners instead of 4 centers once the printed marker orientation
      on the real board is confirmed (more points -> least-squares H, less jitter)
- [ ] undistort with the M1 intrinsics before detection (`calibration.ipynb: undistort`)
- [ ] measure the square-center error on the real board (`run_grid_overlay.ipynb`)

In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path

import chess
import numpy as np
import yaml

%run ./config.ipynb

## Board geometry
Read from `board.yaml` (owned by `gambit_bringup`; on the real board measure with a ruler
and update it there). Marker centers sit `border / 2` outside the nearest playing-area
corner on both axes.

In [ ]:
MARKER_CORNERS = ("a1", "h1", "h8", "a8")


@dataclass
class BoardGeometry:
    square: float        # square edge (m)
    border: float        # frame width around the playing area (m)
    marker_size: float   # marker edge (m)
    dictionary: str      # e.g. "DICT_4X4_50"
    ids: dict            # corner name -> marker id


def load_camera_cfg(path: str | Path = "configs/camera.yaml"):
    return load_config(path)


def load_board_geometry(board_yaml: str | Path | None = None) -> BoardGeometry:
    """Parse gambit_bringup/config/board.yaml (path relative to gambit_vision/)."""
    if board_yaml is None:
        board_yaml = load_camera_cfg().board_yaml
    with open(gv_path(board_yaml)) as f:
        y = yaml.safe_load(f)
    return BoardGeometry(square=float(y["board"]["square"]), border=float(y["board"]["border"]),
                         marker_size=float(y["markers"]["size"]),
                         dictionary=y["markers"]["dictionary"], ids=dict(y["markers"]["ids"]))


def marker_centers_board(geom: BoardGeometry) -> dict[str, np.ndarray]:
    """Marker center (u, v) in board meters for each corner name."""
    s8, h = 8 * geom.square, geom.border / 2
    return {"a1": np.array([-h, -h]), "h1": np.array([s8 + h, -h]),
            "h8": np.array([s8 + h, s8 + h]), "a8": np.array([-h, s8 + h])}

## Board meters <-> warp pixels
An affine map (scale `k = square_px / square`, a flip depending on `view`, an offset of
`margin_px`). Written as a 3x3 matrix so it composes with homographies.

In [ ]:
def warp_size(warp) -> int:
    """Side length (px) of the square top-down image."""
    return 8 * int(warp.square_px) + 2 * int(warp.margin_px)


def board_to_warp_matrix(geom: BoardGeometry, warp) -> np.ndarray:
    """3x3 matrix mapping board meters (u, v, 1) to warp pixels (x, y, 1)."""
    k = warp.square_px / geom.square
    m = float(warp.margin_px)
    full = 8 * warp.square_px
    view = getattr(warp, "view", "black_bottom")
    if view == "white_bottom":   # a-file left, rank 1 bottom
        return np.array([[k, 0, m], [0, -k, m + full], [0, 0, 1]], dtype=float)
    if view == "black_bottom":   # a-file right, rank 8 bottom (camera on the black side)
        return np.array([[-k, 0, m + full], [0, k, m], [0, 0, 1]], dtype=float)
    raise ValueError(f"unknown warp.view {view!r}; use black_bottom or white_bottom")


def _apply(Hm: np.ndarray, pts) -> np.ndarray:
    pts = np.asarray(pts, dtype=float)
    flat = pts.reshape(-1, 2)
    hom = np.c_[flat, np.ones(len(flat))] @ Hm.T
    return (hom[:, :2] / hom[:, 2:3]).reshape(pts.shape)


def board_to_warp(uv, geom: BoardGeometry, warp) -> np.ndarray:
    return _apply(board_to_warp_matrix(geom, warp), uv)


def warp_to_board(xy, geom: BoardGeometry, warp) -> np.ndarray:
    return _apply(np.linalg.inv(board_to_warp_matrix(geom, warp)), xy)

## The 8x8 grid in warp coordinates

In [ ]:
def square_center_board(sq: int, geom: BoardGeometry) -> np.ndarray:
    return (np.array([chess.square_file(sq), chess.square_rank(sq)]) + 0.5) * geom.square


def square_cell_warp(sq: int, geom: BoardGeometry, warp) -> tuple[int, int, int, int]:
    """Axis-aligned cell (x0, y0, x1, y1) of square `sq` in the warp image (x1, y1 exclusive)."""
    f, r = chess.square_file(sq), chess.square_rank(sq)
    corners = board_to_warp(np.array([[f, r], [f + 1, r + 1]]) * geom.square, geom, warp)
    x0, y0 = np.rint(corners.min(axis=0)).astype(int)
    x1, y1 = np.rint(corners.max(axis=0)).astype(int)
    return int(x0), int(y0), int(x1), int(y1)


def square_centers_warp(geom: BoardGeometry, warp) -> np.ndarray:
    """(64, 2) square centers in warp pixels, square-index order."""
    return board_to_warp(np.array([square_center_board(sq, geom) for sq in chess.SQUARES]), geom, warp)


def grid_points_warp(geom: BoardGeometry, warp) -> np.ndarray:
    """(9, 9, 2) grid line intersections in warp pixels, indexed [rank_line, file_line]."""
    r, f = np.meshgrid(np.arange(9), np.arange(9), indexing="ij")
    uv = np.stack([f, r], axis=-1) * geom.square
    return board_to_warp(uv, geom, warp)

## Marker detection and the homography
`cv2.aruco.ArucoDetector` returns 4 corners per marker. The marker center is taken as the
intersection of the two diagonals, which (unlike the mean of the corners) is exactly the
projected center under perspective.

In [ ]:
def get_dictionary(name: str):
    import cv2
    return cv2.aruco.getPredefinedDictionary(getattr(cv2.aruco, name))


def detect_markers(image: np.ndarray, dictionary: str = "DICT_4X4_50") -> dict[int, np.ndarray]:
    """Detect ArUco markers; returns {id: (4, 2) corners in image pixels}."""
    import cv2
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY) if image.ndim == 3 else image
    det = cv2.aruco.ArucoDetector(get_dictionary(dictionary), cv2.aruco.DetectorParameters())
    corners, ids, _ = det.detectMarkers(gray)
    if ids is None:
        return {}
    return {int(i): c.reshape(4, 2).astype(float) for i, c in zip(ids.ravel(), corners)}


def marker_center(corners: np.ndarray) -> np.ndarray:
    """Intersection of the diagonals corner0-corner2 and corner1-corner3."""
    p0, p1, p2, p3 = np.asarray(corners, dtype=float)
    d1, d2 = p2 - p0, p3 - p1
    A = np.array([d1, -d2]).T
    t = np.linalg.solve(A, p1 - p0)[0]
    return p0 + t * d1


def compute_homography(image: np.ndarray, geom: BoardGeometry, warp,
                       markers: dict[int, np.ndarray] | None = None):
    """Homography H mapping image pixels -> warp pixels, from the 4 corner markers.

    Returns (H, info) where info has the detected ids and the image-space marker centers.
    Raises RuntimeError if a corner marker is missing (TODO: 3-marker fallback).
    """
    import cv2
    markers = detect_markers(image, geom.dictionary) if markers is None else markers
    centers_board = marker_centers_board(geom)
    src, dst, missing = [], [], []
    for name in MARKER_CORNERS:
        mid = geom.ids[name]
        if mid not in markers:
            missing.append(name)
            continue
        src.append(marker_center(markers[mid]))
        dst.append(board_to_warp(centers_board[name], geom, warp))
    if missing:
        # TODO M1: with 3 markers use cv2.getAffineTransform or keep the last good H.
        raise RuntimeError(f"corner markers not found: {missing} (detected ids {sorted(markers)})")
    H, _ = cv2.findHomography(np.float32(src), np.float32(dst), 0)
    if H is None:
        raise RuntimeError("findHomography failed (degenerate marker layout?)")
    return H, {"ids": sorted(markers), "centers_image": np.array(src)}


def warp_board(image: np.ndarray, H: np.ndarray, warp) -> np.ndarray:
    """Top-down image of size warp_size(warp) x warp_size(warp)."""
    import cv2
    n = warp_size(warp)
    return cv2.warpPerspective(image, H, (n, n), flags=cv2.INTER_LINEAR)

## Grid model in the camera image, overlay, error metric
The inverse homography maps warp points back into the image. The M1 metric is the pixel
distance between predicted square centers and ground-truth centers (clicked by hand on the
real board, or known exactly in simulation / synthetic images).

In [ ]:
def to_image(points_warp, H: np.ndarray) -> np.ndarray:
    """Map warp-pixel points into image pixels with inv(H)."""
    return _apply(np.linalg.inv(H), points_warp)


def grid_model_image(H: np.ndarray, geom: BoardGeometry, warp) -> dict[str, np.ndarray]:
    """Square centers (64, 2) and grid corners (9, 9, 2) in image pixels."""
    return {"centers": to_image(square_centers_warp(geom, warp), H),
            "corners": to_image(grid_points_warp(geom, warp), H)}


def draw_grid_overlay(image: np.ndarray, H: np.ndarray, geom: BoardGeometry, warp,
                      labels: list[str] | None = None, color=(0, 255, 0),
                      names: bool = True) -> np.ndarray:
    """Copy of `image` with the 8x8 grid, and square names (or 64 labels) drawn on it."""
    import cv2
    out = image.copy()
    g = np.rint(grid_model_image(H, geom, warp)["corners"]).astype(int)
    for i in range(9):
        cv2.polylines(out, [g[i, :, :].reshape(-1, 1, 2)], False, color, 1, cv2.LINE_AA)
        cv2.polylines(out, [g[:, i, :].reshape(-1, 1, 2)], False, color, 1, cv2.LINE_AA)
    if names or labels is not None:
        centers = grid_model_image(H, geom, warp)["centers"]
        for sq, (x, y) in enumerate(centers):
            text = labels[sq] if labels is not None else chess.square_name(sq)
            if text == "empty":
                continue
            cv2.putText(out, text, (int(x) - 8, int(y) + 4), cv2.FONT_HERSHEY_SIMPLEX, 0.3,
                        color, 1, cv2.LINE_AA)
    return out


def square_center_pixel_error(pred: np.ndarray, true: np.ndarray) -> dict[str, float]:
    """Euclidean error between predicted and true square centers (N, 2): mean, rms, max (px)."""
    d = np.linalg.norm(np.asarray(pred, float) - np.asarray(true, float), axis=-1)
    return {"mean": float(d.mean()), "rms": float(np.sqrt((d ** 2).mean())), "max": float(d.max())}

## Synthetic board (for the self-check and for debugging without a camera)
Draws the board in warp coordinates: frame, light/dark squares and the four markers, each
with a one-module white quiet zone (ArUco detection needs it; print them that way too).

In [ ]:
def render_synthetic_board(geom: BoardGeometry, warp, light=(190, 220, 235), dark=(60, 85, 120),
                           frame=(35, 50, 70)) -> np.ndarray:
    """BGR top-down board image in warp coordinates (size warp_size(warp))."""
    import cv2
    n = warp_size(warp)
    img = np.full((n, n, 3), 255, np.uint8)
    k = warp.square_px / geom.square
    # frame (playing area + border)
    lo = board_to_warp(np.array([[-geom.border, -geom.border],
                                 [8 * geom.square + geom.border] * 2]), geom, warp)
    (x0, y0), (x1, y1) = np.rint(lo.min(axis=0)).astype(int), np.rint(lo.max(axis=0)).astype(int)
    cv2.rectangle(img, (x0, y0), (x1, y1), frame, -1)
    for sq in chess.SQUARES:
        cx0, cy0, cx1, cy1 = square_cell_warp(sq, geom, warp)
        is_light = (chess.square_file(sq) + chess.square_rank(sq)) % 2 == 1
        img[cy0:cy1, cx0:cx1] = light if is_light else dark
    side = int(round(geom.marker_size * k))
    quiet = int(round(side / 6))  # 4x4 marker = 6 modules incl. its black border
    dictionary = get_dictionary(geom.dictionary)
    for name, uv in marker_centers_board(geom).items():
        cx, cy = np.rint(board_to_warp(uv, geom, warp)).astype(int)
        x, y = cx - side // 2, cy - side // 2
        if x - quiet < 0 or y - quiet < 0 or x + side + quiet > n or y + side + quiet > n:
            raise ValueError("marker outside the warp image; increase warp.margin_px")
        img[y - quiet:y + side + quiet, x - quiet:x + side + quiet] = 255
        m = cv2.aruco.generateImageMarker(dictionary, geom.ids[name], side)
        img[y:y + side, x:x + side] = m[:, :, None]
    return img

## Self-check
Render a synthetic board, push it through a known perspective (the "camera"), recover the
homography from the detected markers and compare predicted square centers with the truth.

In [ ]:
from types import SimpleNamespace

import cv2

_hg_geom = load_board_geometry()
assert _hg_geom.ids == {"a1": 0, "h1": 1, "h8": 2, "a8": 3}
assert abs(_hg_geom.square - 0.025) < 1e-9
_hg_cam = load_camera_cfg()
assert _hg_cam.markers.ids.__dict__ == _hg_geom.ids, "camera.yaml markers.ids != board.yaml"

# frame conventions: black_bottom puts a1 top-right, h8 bottom-left
_hg_w = SimpleNamespace(square_px=60, margin_px=90, view="black_bottom")
_c = square_centers_warp(_hg_geom, _hg_w)
assert np.allclose(_c[chess.A1], [90 + 7.5 * 60, 90 + 0.5 * 60])
assert np.allclose(_c[chess.H8], [90 + 0.5 * 60, 90 + 7.5 * 60])
assert np.allclose(warp_to_board(board_to_warp([[0.01, 0.02]], _hg_geom, _hg_w), _hg_geom, _hg_w), [[0.01, 0.02]])
_ww = SimpleNamespace(square_px=60, margin_px=90, view="white_bottom")
assert np.allclose(square_centers_warp(_hg_geom, _ww)[chess.A1], [90 + 30, 90 + 7.5 * 60])
assert square_cell_warp(chess.A1, _hg_geom, _hg_w) == (90 + 420, 90, 90 + 480, 150)
assert np.allclose(marker_center(np.array([[0, 0], [2, 0], [2, 2], [0, 2]])), [1, 1])

# synthetic camera view
_board_img = render_synthetic_board(_hg_geom, _hg_w)
_n = warp_size(_hg_w)
_H_true = cv2.getPerspectiveTransform(  # warp -> camera image
    np.float32([[0, 0], [_n, 0], [_n, _n], [0, _n]]),
    np.float32([[120, 40], [520, 50], [615, 455], [25, 445]]))
_cam = cv2.warpPerspective(_board_img, _H_true, (640, 480), borderValue=(128, 128, 128))
_H_est, _info = compute_homography(_cam, _hg_geom, _hg_w)
assert _info["ids"] == [0, 1, 2, 3]
_true_centers = _apply(_H_true, square_centers_warp(_hg_geom, _hg_w))
_err = square_center_pixel_error(grid_model_image(_H_est, _hg_geom, _hg_w)["centers"], _true_centers)
assert _err["max"] < 1.5, _err
_overlay = draw_grid_overlay(_cam, _H_est, _hg_geom, _hg_w)
assert _overlay.shape == _cam.shape and not np.array_equal(_overlay, _cam)
assert warp_board(_cam, _H_est, _hg_w).shape == (_n, _n, 3)

# a covered marker is reported
_cam2 = _cam.copy()
_x, _y = np.rint(_info["centers_image"][0]).astype(int)
_cam2[_y - 30:_y + 30, _x - 30:_x + 30] = 128
try:
    compute_homography(_cam2, _hg_geom, _hg_w)
    raise AssertionError("expected RuntimeError")
except RuntimeError as e:
    assert "a1" in str(e)
print("homography OK, synthetic square-center error (px):", {k: round(v, 3) for k, v in _err.items()})